# 04 — Continued Pretraining of mT5-small (Kaggle T4 / P100)

Phase 2 of TauraBot, adapted for Kaggle Notebooks.

**Why Kaggle here:** 30 GPU-hours/week of T4×2 or P100 + 9-hour session length comfortably fits the planned 10,000 training steps (~2–3 hours on a single T4) in one session, with budget left over for fine-tuning.

---
## One-time setup (do this on Kaggle's website, before opening the notebook)

**1. Create two Kaggle Datasets** (top-right *Create → New Dataset*):

| Dataset slug | What to upload | Approx size |
|---|---|---|
| `taurabot-code` | The whole TauraBot folder **minus** `venv/`, `data/raw/`, `data/processed/`, `__pycache__/` | < 1 MB |
| `taurabot-corpus` | Just `data/processed/corpus.txt` | 360 MB |

Mark both **private** (you can change later before the public model release).

**2. Add your HuggingFace token as a Kaggle Secret** (notebook → Add-ons → Secrets):
- Label: `HF_TOKEN`
- Value: write-scoped token from <https://huggingface.co/settings/tokens>
- Toggle it **on for this notebook**

**3. Attach the two datasets to this notebook** (right sidebar → Add Input → Datasets) and enable **GPU T4×2** under *Notebook options*.

**4. Verify the input paths below match what Kaggle gave you** — Kaggle puts each attached dataset at `/kaggle/input/<slug>/`. Slugs are kebab-case versions of dataset titles.

In [ ]:
# ---- adjust these to match your Kaggle dataset slugs ----
CODE_DATASET_PATH   = '/kaggle/input/taurabot-code'
CORPUS_DATASET_PATH = '/kaggle/input/taurabot-corpus'
# Optional: attach this if resuming a prior session's checkpoints
PRIOR_CKPT_PATH     = '/kaggle/input/taurabot-checkpoints'   # may not exist on first run

# Where this session works (writable, persists at session end)
WORK = '/kaggle/working'
OUTPUT_DIR = f'{WORK}/checkpoints/shona-mt5-small'

# HuggingFace Hub target for the final model
HUB_MODEL_ID = 'YOUR-USERNAME/shona-mt5-small'   # CHANGE ME

## 1. Verify GPU + inputs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import os
for p in (CODE_DATASET_PATH, CORPUS_DATASET_PATH):
    print(f"{'OK' if os.path.exists(p) else 'MISSING':>8}  {p}")
!ls $CODE_DATASET_PATH 2>/dev/null | head -10
!ls -lh $CORPUS_DATASET_PATH 2>/dev/null

## 2. Stage the code + corpus into /kaggle/working

`/kaggle/input/` is read-only and Python's `datasets` library wants to write a cache, so we copy the code into the writable working dir. The corpus is read-only — fine — but we symlink it into the expected project path.

In [ ]:
import os, shutil

PROJECT_DIR = f'{WORK}/taurabot'
if not os.path.exists(PROJECT_DIR):
    shutil.copytree(CODE_DATASET_PATH, PROJECT_DIR)

# Place corpus where pretrain.py expects to find it (configs/pretrain.yaml default)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
corpus_link = f'{PROJECT_DIR}/data/processed/corpus.txt'
if not os.path.exists(corpus_link):
    # Find corpus.txt inside the corpus dataset (any subfolder)
    import glob
    matches = glob.glob(f'{CORPUS_DATASET_PATH}/**/corpus.txt', recursive=True)
    assert matches, f'corpus.txt not found inside {CORPUS_DATASET_PATH}'
    os.symlink(matches[0], corpus_link)

# Optional: restore prior-session checkpoints if a checkpoints dataset is attached
if os.path.exists(PRIOR_CKPT_PATH):
    print(f'Found prior checkpoints at {PRIOR_CKPT_PATH}, copying to {OUTPUT_DIR}...')
    os.makedirs(os.path.dirname(OUTPUT_DIR), exist_ok=True)
    if not os.path.exists(OUTPUT_DIR):
        shutil.copytree(PRIOR_CKPT_PATH, OUTPUT_DIR)
    print(f'  resumed checkpoint dirs: {sorted(os.listdir(OUTPUT_DIR))}')
else:
    print('No prior checkpoints — fresh training run.')

!ls $PROJECT_DIR
!wc -l $corpus_link

## 3. Install training dependencies

Kaggle's base image already has CUDA-enabled PyTorch. We just need recent versions of `transformers`/`datasets`/`accelerate` + the SentencePiece dep that mT5's tokenizer requires.

In [ ]:
!pip install -q -U 'transformers>=4.40,<5.0' 'accelerate>=0.30,<1.0' \
                   'datasets>=2.18,<3.0' 'huggingface_hub>=0.24,<1.0' \
                   'sentencepiece>=0.1.99' 'protobuf>=3.20,<5.0' \
                   'evaluate>=0.4' pyyaml tensorboard
import torch; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'n_gpus:', torch.cuda.device_count())

## 4. HuggingFace Hub login via Kaggle Secrets

Uses the `HF_TOKEN` secret you attached during one-time setup.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
print('Hugging Face Hub login OK.')

## 5. Run pretraining

Drives `src/model/pretrain.py` with config + CLI overrides for Kaggle paths and Hub push. First run: tokenization ~5 min (cached afterwards), then training ~2–3 hours on T4 for 10,000 steps. Auto-resumes from any checkpoint in `OUTPUT_DIR`.

**If you only have ~1 hour left in this session**, lower `--max_steps` (e.g. 3000) — the next session can resume.

In [ ]:
import subprocess, os

# All CLI flags here OVERRIDE the same fields in configs/pretrain.yaml — keeps Kaggle-
# specific paths out of the version-controlled YAML.
cmd = [
    'python', '-m', 'src.model.pretrain',
    '--config', 'configs/pretrain.yaml',
    '--corpus_path', f'{PROJECT_DIR}/data/processed/corpus.txt',
    '--output_dir', OUTPUT_DIR,
    '--cache_dir', f'{WORK}/tokenized_cache',
    '--push_to_hub', 'true',
    '--hub_model_id', HUB_MODEL_ID,
]
print('Running:', ' '.join(cmd))
os.chdir(PROJECT_DIR)
subprocess.run(cmd, check=True)

## 6. (If you ran out of session) Save checkpoints as a Kaggle Dataset for resume

Manual step: Kaggle → *Save Version* (top-right) with **Save & Run All** unchecked, then use the persisted output as a new dataset. Attach it next session via the `PRIOR_CKPT_PATH` variable at the top of this notebook.

Alternative (smoother): training already pushed to HuggingFace Hub at the end of step 5 via `--push_to_hub`. Next session, set `OUTPUT_DIR` to a fresh dir and prepend a cell that pulls weights from Hub:

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
AutoModelForSeq2SeqLM.from_pretrained(HUB_MODEL_ID).save_pretrained(OUTPUT_DIR)
AutoTokenizer.from_pretrained(HUB_MODEL_ID).save_pretrained(OUTPUT_DIR)
```

Note: the Hub-only approach loses optimizer/scheduler state, so step counter restarts from 0 — fine for short additional runs, suboptimal for picking up mid-training. Use the Kaggle-Dataset-snapshot path if you care about optimizer state.

## 7. Sanity-check the trained model with a span infill

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
mdl = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to('cuda').eval()

# mT5's <extra_id_0> sentinel = vocab_size - 1
sentinel_id = tok.vocab_size - 1

prompts = [
    'Mhoro, ndinodaidzwa kuti <extra_id_0>',          # Hello, my name is ___
    'Pakutanga Mwari akasika <extra_id_0> nenyika',    # In the beginning God created ___ and earth
    'Zita rangu ndiTaura uye ndinotaura <extra_id_0>', # My name is Taura and I speak ___
]
# Some tokenizers don't tokenize '<extra_id_0>' to the right ID by default. Patch
# by tokenizing manually: split on the sentinel string and concatenate ids around it.
def encode_with_sentinel(text: str) -> torch.Tensor:
    parts = text.split('<extra_id_0>')
    ids = []
    for i, part in enumerate(parts):
        ids.extend(tok(part, add_special_tokens=False)['input_ids'])
        if i < len(parts) - 1:
            ids.append(sentinel_id)
    ids.append(tok.eos_token_id)
    return torch.tensor([ids]).to('cuda')

for p in prompts:
    enc = encode_with_sentinel(p)
    out = mdl.generate(input_ids=enc, max_new_tokens=30, num_beams=4)
    print(f'\nIN:  {p}')
    print(f'OUT: {tok.decode(out[0], skip_special_tokens=False)}')

## What's next

If the loss looks reasonable (mT5-small ≈ 4.5 → 3.0 after 10k steps on a Shona corpus this size) and the span infills produce plausible Shona, you have a working `shona-mt5-small`.

Write the **model card** on the Hub page (training data summary, hyperparameters, intended use, limitations). The corpus README block from notebook 03 is a good starting template.

Next: Phase 3 — conversation fine-tuning on hand-crafted Shona Q&A pairs.